In [1]:
"""Phase G: Gegenproben zum LightGBM-Modell, Bericht 3.9.
Ausführen: uv run python -u src/counter_tests.py   (dauert ca. 15 Minuten; Ausgabe: reports/counter_tests_output.txt, reports/results_counter_tests.csv)
Vorher: src/clean.py, src/baselines.py und src/gbm.py müssen gelaufen sein.
"""

'Phase G: Gegenproben zum LightGBM-Modell, Bericht 3.9.\nAusführen: uv run python -u src/counter_tests.py   (dauert ca. 15 Minuten; Ausgabe: reports/counter_tests_output.txt, reports/results_counter_tests.csv)\nVorher: src/clean.py, src/baselines.py und src/gbm.py müssen gelaufen sein.\n'

In [2]:
import sys
import time
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd().resolve().parent  # Skript oder Notebook
sys.path.insert(0, str(ROOT / "src"))
from forecast_eval import WINDOWS, load_clean, score, window_frames  # noqa: E402
from gbm_model import DEFAULT_DROP, Panel, fit_predict  # noqa: E402
from masking import spike_flags  # noqa: E402

pd.set_option("display.width", 170)
lines = []


def out(msg=""):
    print(msg, flush=True)
    lines.append(str(msg))


df = load_clean()

## 1. Was wird getestet?
Jede Gegenprobe ändert **eine** Sache gegenüber dem Basismodell (Bericht 3.8) und rechnet alle drei Fenster neu.
Bewertet wird immer auf denselben Zeilen mit demselben Rohwert `sales`, nur das Training ändert sich.

| Gruppe | Frage |
|---|---|
| Rauschen | Wie stark schwankt das Ergebnis, wenn nur die Zufallszahlen anders sind? Das ist die Messlatte für alles andere. |
| Merkmale weglassen | Welche Merkmalsgruppe zieht wirklich Gewicht? |
| Ölpreis | Hilft der Ölpreis (am Ursprung bekannt)? |
| `promo_known` | Hilft es, nur mit verlässlichen Aktionsangaben zu trainieren? |
| Maskierung | Spielt es eine Rolle, ob und wie streng Spitzen maskiert werden? |

In [3]:
GROUPS = {
    "aktion": ("aktion", "aktion_bekannt"),
    "kalender_feiertag": ("feiertag", "ereignis", "arbeitstag_ausgleich"),
    "monat": ("monat", "tag_im_monat"),
    "filiale_familie": ("family", "store", "type", "city", "state", "cluster"),
}
ALL_GROUPS = tuple(c for g in GROUPS.values() for c in g)
VARIANTS = [
    ("Basis", {}),
    ("Basis, andere Zufallszahlen", {"seed": 1}),
    ("ohne Aktion", {"drop": DEFAULT_DROP + GROUPS["aktion"]}),
    ("ohne Feiertage und Ereignisse", {"drop": DEFAULT_DROP + GROUPS["kalender_feiertag"]}),
    ("ohne Monat und Tag im Monat", {"drop": DEFAULT_DROP + GROUPS["monat"]}),
    ("ohne Filial- und Familienmerkmale", {"drop": DEFAULT_DROP + GROUPS["filiale_familie"]}),
    ("nur Verkaufshistorie und Wochentag", {"drop": DEFAULT_DROP + ALL_GROUPS}),
    ("mit Ölpreis", {"drop": ()}),
    ("Training nur mit promo_known = 1", {"only_promo_known": True}),
]
MASKING = [("keine Maskierung der Spitzen", None), ("Maskierung Faktor 5", 5)]

truths = {name: window_frames(df, origin) for name, origin in WINDOWS.items()}
panel = Panel(df)
rows, fam_w1 = [], {}


def run(label, pnl, **kw):
    t0 = time.time()
    vals, fam = {}, None
    for name, origin in WINDOWS.items():
        pred, _, _ = fit_predict(pnl, origin, **kw)
        truth, excluded, outlier = truths[name]
        r = score(truth, pred, excluded, outlier)
        vals[name] = r["rmsle"]
        if name.startswith("W1"):
            fam = r["je_familie"]
    fam_w1[label] = fam
    rows.append({"Variante": label, **vals})
    out(f"{label}: " + ", ".join(f"{k.split()[0]} {v:.4f}" for k, v in vals.items()) + f"  ({time.time() - t0:.0f} s)")


for label, kw in VARIANTS:
    run(label, panel, **kw)

Basis: W1 0.3939, W2 0.4358, W3 0.3987  (79 s)


Basis, andere Zufallszahlen: W1 0.3954, W2 0.4392, W3 0.4011  (93 s)


ohne Aktion: W1 0.4580, W2 0.4393, W3 0.4262  (125 s)


ohne Feiertage und Ereignisse: W1 0.3976, W2 0.4338, W3 0.3992  (109 s)


ohne Monat und Tag im Monat: W1 0.4016, W2 0.5074, W3 0.3980  (112 s)


ohne Filial- und Familienmerkmale: W1 0.4145, W2 0.5135, W3 0.4204  (92 s)


nur Verkaufshistorie und Wochentag: W1 0.4889, W2 0.5542, W3 0.4128  (83 s)


mit Ölpreis: W1 0.3966, W2 0.4397, W3 0.4027  (116 s)


Training nur mit promo_known = 1: W1 0.3949, W2 0.4358, W3 0.3987  (116 s)


## 2. Maskierung der Spitzen
Hier ändert sich das Training: Die Spalte `sales_train` wird mit einer anderen Regel maskiert (Bericht 2.9).
Geschlossene Tage und Filialen bleiben in allen Varianten maskiert. Bewertet wird weiter auf dem Rohwert.

In [4]:
closed = (df.is_closed_day == 1) | (df.is_store_closed == 1)
for label, factor in MASKING:
    d2 = df.copy()
    spike = spike_flags(df, factor) if factor else pd.Series(False, index=df.index)
    d2["sales_train"] = df.sales.where(~(closed | spike))
    out(f"[{label}] maskierte Spitzen: {int(spike.sum())}")
    run(label, Panel(d2))

[keine Maskierung der Spitzen] maskierte Spitzen: 0


keine Maskierung der Spitzen: W1 0.3965, W2 0.4334, W3 0.3997  (119 s)


[Maskierung Faktor 5] maskierte Spitzen: 708


Maskierung Faktor 5: W1 0.3953, W2 0.4372, W3 0.3998  (116 s)


## 3. Ergebnis
`Änderung` = Mittel der drei Fenster gegenüber dem Basismodell. Positiv = schlechter. Das Rauschen steht in der Zeile
„Basis, andere Zufallszahlen“: Änderungen in dieser Größe sind nicht von Zufall zu unterscheiden.

In [5]:
res = pd.DataFrame(rows).set_index("Variante")
res["Mittel"] = res[list(WINDOWS)].mean(axis=1)
res["Änderung"] = (res.Mittel / res.loc["Basis", "Mittel"] - 1).map("{:+.1%}".format)
for w in WINDOWS:
    res[f"Δ {w.split()[0]}"] = (res[w] / res.loc["Basis", w] - 1).map("{:+.1%}".format)
out("\n" + res.round(4).to_string())
noise = abs(res.loc["Basis, andere Zufallszahlen", "Mittel"] / res.loc["Basis", "Mittel"] - 1)
out(f"\nRauschen (Mittel der drei Fenster, anderer Zufall): {noise:.2%}")


                                    W1 Aug 2017  W2 Dez 2016  W3 Mär 2017  Mittel Änderung    Δ W1    Δ W2   Δ W3
Variante                                                                                                         
Basis                                    0.3939       0.4358       0.3987  0.4095    +0.0%   +0.0%   +0.0%  +0.0%
Basis, andere Zufallszahlen              0.3954       0.4392       0.4011  0.4119    +0.6%   +0.4%   +0.8%  +0.6%
ohne Aktion                              0.4580       0.4393       0.4262  0.4412    +7.7%  +16.3%   +0.8%  +6.9%
ohne Feiertage und Ereignisse            0.3976       0.4338       0.3992  0.4102    +0.2%   +0.9%   -0.5%  +0.1%
ohne Monat und Tag im Monat              0.4016       0.5074       0.3980  0.4357    +6.4%   +1.9%  +16.4%  -0.2%
ohne Filial- und Familienmerkmale        0.4145       0.5135       0.4204  0.4495    +9.8%   +5.2%  +17.8%  +5.4%
nur Verkaufshistorie und Wochentag       0.4889       0.5542       0.4128  0.4853   +18


Rauschen (Mittel der drei Fenster, anderer Zufall): 0.59%


## 4. Welche Familien leiden, wenn eine Merkmalsgruppe fehlt? (Fenster W1)

In [6]:
base = pd.Series(fam_w1["Basis"])
for label in ["ohne Aktion", "ohne Feiertage und Ereignisse", "ohne Monat und Tag im Monat", "ohne Filial- und Familienmerkmale"]:
    d = (pd.Series(fam_w1[label]) / base - 1).sort_values(ascending=False)
    out(f"\n{label}: am stärksten verschlechtert (W1)")
    out(d.head(5).map("{:+.1%}".format).to_string())
    out(f"Familien verschlechtert um mehr als 2 %: {int((d > 0.02).sum())} von {len(d)}")


ohne Aktion: am stärksten verschlechtert (W1)


SCHOOL AND OFFICE SUPPLIES    +110.4%
BEAUTY                         +11.9%
BOOKS                          +11.0%
LINGERIE                        +6.4%
LIQUOR,WINE,BEER                +5.9%


Familien verschlechtert um mehr als 2 %: 8 von 33



ohne Feiertage und Ereignisse: am stärksten verschlechtert (W1)


PRODUCE          +9.1%
DAIRY            +7.0%
BOOKS            +6.6%
FROZEN FOODS     +6.0%
PERSONAL CARE    +4.6%


...

PERSONAL CARE    +33.6%
GROCERY I        +30.4%
CLEANING         +15.0%
DAIRY            +12.9%
HOME CARE        +12.5%


Familien verschlechtert um mehr als 2 %: 17 von 33



ohne Filial- und Familienmerkmale: am stärksten verschlechtert (W1)


BOOKS                         +36.4%
DAIRY                         +21.7%
GROCERY I                     +17.7%
PERSONAL CARE                 +16.2%
SCHOOL AND OFFICE SUPPLIES    +16.1%


Familien verschlechtert um mehr als 2 %: 27 von 33


In [7]:
res.to_csv(ROOT / "reports" / "results_counter_tests.csv")
(ROOT / "reports" / "counter_tests_output.txt").write_text("\n".join(lines), encoding="utf-8")

3420